# PDM — browse projects, libraries, files

Projects and libraries are sibling collections on a safe; each is
an `IADPDMFolder`, so the same browsing code works for both.

**Prereq:** same as `pdm_quickstart.ipynb` — log in via the UI, or
edit the connection constants in the first code cell.

## Connect

In [ ]:
from alibrex import connect

# === EDIT IF NEEDED =====================================
PDM_URL      = "http://localhost:8099/"
PDM_DOMAIN   = ""
PDM_USER     = ""
PDM_PASSWORD = ""
# ========================================================

root = connect()
try:
    conn = root.GetActiveServerConnection()
    if conn is None:
        raise RuntimeError("no active session")
except Exception:
    print("  No active connection — connecting directly ...")
    try:
        conn = root.ConnectToPDM(PDM_URL, PDM_DOMAIN, PDM_USER, PDM_PASSWORD)
    except Exception as exc:
        print(f"  ConnectToPDM failed: {exc}")
        raise SystemExit(1)

safe = conn.Safes.Item(0)
safe.Name

## List projects

In [ ]:
projects = safe.Projects
print(f"Projects: {projects.Count}")
for i in range(projects.Count):
    p = projects.Item(i)
    print(f"  [{i}] {p.Name!r}  files={p.FileItems.Count}  subfolders={p.Folders.Count}")

## List libraries

In [ ]:
libs = safe.Libraries
print(f"Libraries: {libs.Count}")
for i in range(libs.Count):
    l = libs.Item(i)
    print(f"  [{i}] {l.Name!r}  files={l.FileItems.Count}  subfolders={l.Folders.Count}")

## Walk the first project's tree (depth-limited)

An iterative DFS keeps the loop function-less. Track depth so deeply
nested trees don't drown the output.

In [ ]:
MAX_DEPTH = 2

proj = projects.Item(0)
stack = [(0, proj.Name, proj)]
while stack:
    depth, path, folder = stack.pop()
    indent = "  " * depth
    files = folder.FileItems
    print(f"{indent}{path}/")
    for i in range(files.Count):
        fi = files.Item(i)
        lock = " [LOCKED]" if fi.IsLocked else ""
        print(f"{indent}  - {fi.Name}.{fi.Extension}  v{fi.CurrentVersionID}{lock}")
    if depth < MAX_DEPTH:
        subs = folder.Folders
        for i in range(subs.Count):
            sub = subs.Item(i)
            stack.append((depth + 1, f"{path}/{sub.Name}", sub))

## Find the first file in the project — at any depth

Most projects don't have files directly at the root; they're inside
subfolders. This iterative DFS walks down until it finds a folder
with at least one file.

In [ ]:
target_file = None
target_path = ""
stack = [(proj.Name, proj)]
while stack and target_file is None:
    path, folder = stack.pop()
    if folder.FileItems.Count > 0:
        target_file = folder.FileItems.Item(0)
        target_path = path
    else:
        for i in range(folder.Folders.Count):
            sub = folder.Folders.Item(i)
            stack.append((f"{path}/{sub.Name}", sub))

if target_file is None:
    print(f"Project {proj.Name!r} has no files at any depth.")
else:
    print(f"Found: {target_path}/{target_file.Name}.{target_file.Extension}  v{target_file.CurrentVersionID}")

## Properties on that file

In [ ]:
if target_file is None:
    print("No file to inspect.")
else:
    props = target_file.Properties
    print(f"Properties: {props.Count}")
    for p in range(props.Count):
        prop = props.Item(p)
        val  = prop.Value if prop.HasValue else "(empty)"
        print(f"  {prop.DisplayName!r:30s}  =  {val}")

## Version history on that file

In [ ]:
if target_file is None:
    print("No file to inspect.")
else:
    history = target_file.History
    print(f"File: {target_path}/{target_file.Name}.{target_file.Extension}")
    print(f"History entries: {history.Count}")
    for h in range(min(history.Count, 5)):
        ver = history.Item(h)
        rev = f" rev:{ver.Revision}" if ver.Revision else ""
        print(f"  v{ver.Version}  {ver.CheckedInBy}  {ver.CheckedInAt}{rev}")
        if ver.VersionComment:
            print(f"        {ver.VersionComment!r}")